# eeg2img on Colab: data preparation

This notebook only launches the repository's scripts; it contains no logic of its own.
It downloads Alljoined-1.6M, preprocesses it into a compact store and computes the frozen
DINOv2-giant image targets. Allow about 1 hour on a T4 and 20 GB of Colab disk.

**Before running:** Runtime → Change runtime type → **T4 GPU**. While the repository is
private, add a GitHub token with read-only *Contents* access as the Colab secret `GH_TOKEN`.


In [ ]:
REPO = "moh246/eeg2image-neurips26"
import os, subprocess
if not os.path.isdir("/content/eeg2img"):
    url = f"https://github.com/{REPO}.git"
    try:  # only needed while the repository is private
        from google.colab import userdata
        url = f"https://{userdata.get('GH_TOKEN')}@github.com/{REPO}.git"
    except Exception:
        pass
    subprocess.run(["git", "clone", "--quiet", url, "/content/eeg2img"], check=True)
%cd /content/eeg2img
!git log -1 --format='%h %s'
%pip install --quiet -e ".[data]"


## Build the store

Alljoined-1.6M is licensed **CC-BY-NC-ND-4.0** (non-commercial); running the next cell accepts
that licence. Every step resumes if the session is interrupted. The second argument is where
the finished store (3.7 GB) is copied: keep `/content/persist` to stay on the Colab disk, or
mount Drive and point it there if you have the space and want it to survive the session.


In [ ]:
!bash scripts/colab_prepare.sh /content/work /content/persist


## Check it

The SHA-256 sums printed above should match `data/README.md` (`embeddings.npy` may differ in
the last bits across GPUs). The next cell checks the image-group assumption of the split.


In [ ]:
STORE = "/content/work/stores/alljoined16m_v1"
!python scripts/check_image_groups.py --config configs/experiment/alljoined_conv.yaml \
    data.store={STORE} data.embeddings={STORE}/embeddings.npy
